## Ejercicio 2

Se buscará resolver la clasificación de los dígitos de MNIST usando la siguiente configuración:

```python
model = Sequential()
model.add(Input(shape=(28, 28, 1)))
model.add(Conv2D(F, kernel_size=K, strides=(S,S), activation=FUN))
model.add(MaxPooling2D(pool_size=(2,2)))   # -- opcional --
model.add(Flatten())
model.add(Dense(10,activation='softmax'))

model.summary()
```

Donde **F** es la cantidad de filtros o de mapas de características, **K** es el tamaño del kernel o máscara, **S** es el valor del stride y **FUN** es la función de activación de la capa de convolución.

La tabla que aparece a continuación sugiere los valores a utilizar. Se recomienda emplear Parada Temprana para reducir el tiempo de entrenamiento. Para ello utilice:

```python
from tensorflow.keras.callbacks import EarlyStopping  
es = EarlyStopping(monitor='val_accuracy', patience=5, min_delta=0.001)
```

Esto indica que, si el valor del accuracy sobre los datos de validación no mejora después de 5 épocas, el entrenamiento finaliza. Puede usarse el parámetro min_delta para indicar cuando la diferencia entre dos accuracy se considerará significativa. Luego agregue este objeto en el momento del entrenamiento por medio del párametro callbacks

```python
H = model.fit(x = X_train, y = Y_train, batch_size = LOTES,
              validation_data = (X_test, Y_test), epochs = 4000, callbacks = [es])
```

<p align="center">

<table>
<tr><th>Cant. de filtros</th><th>Tamaño del kernel o filtro</th><th>Stride</th><th>Función de activación</th><th>Max Pooling</th><th>Total de parámetros</th><th>Épocas</th><th>Accuracy Train</th><th>Accuracy Test</th></tr>
<tr><td>4</td><td>3x3</td><td>1</td><td>ReLU</td><td>Sí</td><td></td><td></td><td></td><td></td></tr>
<tr><td>16</td><td>3x3</td><td>1</td><td>ReLU</td><td>Sí</td><td></td><td></td><td></td><td></td></tr>
<tr><td>64</td><td>3x3</td><td>1</td><td>ReLU</td><td>Sí</td><td></td><td></td><td></td><td></td></tr>
<tr><td>4</td><td>7x7</td><td>1</td><td>ReLU</td><td>Sí</td><td></td><td></td><td></td><td></td></tr>
<tr><td>16</td><td>7x7</td><td>1</td><td>ReLU</td><td>Sí</td><td></td><td></td><td></td><td></td></tr>
<tr><td>64</td><td>7x7</td><td>1</td><td>ReLU</td><td>Sí</td><td></td><td></td><td></td><td></td></tr>
<tr><td>64</td><td>3x3</td><td>2</td><td>ReLU</td><td>No</td><td></td><td></td><td></td><td></td></tr>
<tr><td>64</td><td>3x3</td><td>3</td><td>ReLU</td><td>No</td><td></td><td></td><td></td><td></td></tr>
<tr><td>64</td><td>3x3</td><td>1</td><td>TanH</td><td>Sí</td><td></td><td></td><td></td><td></td></tr>
<tr><td>64</td><td>3x3</td><td>1</td><td>Sigmoide</td><td>Sí</td><td></td><td></td><td></td><td></td></tr>
</table>

</p>



In [3]:
from tensorflow.keras.datasets import mnist
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Dense, Flatten, Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping
import numpy as np
import tensorflow as tf
import pandas as pd
from tensorflow.keras.models import load_model
DATOS_DIR = 'Modelos/'

import  matplotlib.pyplot as plt
import numpy as np

In [5]:

# === Carga y preprocesamiento de MNIST ===
(X_train, Y_train), (X_test, Y_test) = mnist.load_data()

# Normalización y reshaping
X_train = X_train.astype('float32') / 255.0
X_test  = X_test.astype('float32') / 255.0
X_train = np.expand_dims(X_train, -1)
X_test  = np.expand_dims(X_test, -1)

# One-hot encoding
Y_train = to_categorical(Y_train, 10)
Y_test  = to_categorical(Y_test, 10)

# === Early stopping ===
es = EarlyStopping(monitor='val_accuracy', patience=5, min_delta=0.001, restore_best_weights=True)

# === Configuraciones a testear ===
configuraciones = [
    {"F":4,  "K":(3,3), "S":1, "FUN":"relu",     "POOL":True},
    {"F":16, "K":(3,3), "S":1, "FUN":"relu",     "POOL":True},
    {"F":64, "K":(3,3), "S":1, "FUN":"relu",     "POOL":True},
    {"F":4,  "K":(7,7), "S":1, "FUN":"relu",     "POOL":True},
    {"F":16, "K":(7,7), "S":1, "FUN":"relu",     "POOL":True},
    {"F":64, "K":(7,7), "S":1, "FUN":"relu",     "POOL":True},
    {"F":64, "K":(3,3), "S":2, "FUN":"relu",     "POOL":False},
    {"F":64, "K":(3,3), "S":3, "FUN":"relu",     "POOL":False},
    {"F":64, "K":(3,3), "S":1, "FUN":"tanh",     "POOL":True},
    {"F":64, "K":(3,3), "S":1, "FUN":"sigmoid",  "POOL":True},
]

# === Entrenamiento automático ===
resultados = []

for i, cfg in enumerate(configuraciones, start=1):
    print(f"\n🧩 Entrenando modelo {i}/10 → {cfg}")
    F, K, S, FUN, POOL = cfg.values()

    # Definición del modelo
    model = Sequential()
    model.add(Input(shape=(28, 28, 1)))
    model.add(Conv2D(F, kernel_size=K, strides=(S, S), activation=FUN))
    if POOL:
        model.add(MaxPooling2D(pool_size=(2, 2)))
    model.add(Flatten())
    model.add(Dense(10, activation='softmax'))

    # Compilación
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

    # Entrenamiento
    H = model.fit(
        X_train, Y_train,
        batch_size=128,
        validation_data=(X_test, Y_test),
        epochs=4000,
        callbacks=[es],
        verbose=0
    )

    # Evaluación
    train_acc = H.history['accuracy'][-1]
    test_loss, test_acc = model.evaluate(X_test, Y_test, verbose=0)

    resultados.append({
        "Filtros": F,
        "Kernel": f"{K[0]}x{K[1]}",
        "Stride": S,
        "Activación": FUN,
        "MaxPooling": "Sí" if POOL else "No",
        "Parámetros": model.count_params(),
        "Épocas": len(H.history['accuracy']),
        "Acc Train": round(train_acc, 4),
        "Acc Test": round(test_acc, 4)
    })

# === Mostrar resultados como tabla ===
df_resultados = pd.DataFrame(resultados)
display(df_resultados)


🧩 Entrenando modelo 1/10 → {'F': 4, 'K': (3, 3), 'S': 1, 'FUN': 'relu', 'POOL': True}

🧩 Entrenando modelo 2/10 → {'F': 16, 'K': (3, 3), 'S': 1, 'FUN': 'relu', 'POOL': True}

🧩 Entrenando modelo 3/10 → {'F': 64, 'K': (3, 3), 'S': 1, 'FUN': 'relu', 'POOL': True}

🧩 Entrenando modelo 4/10 → {'F': 4, 'K': (7, 7), 'S': 1, 'FUN': 'relu', 'POOL': True}

🧩 Entrenando modelo 5/10 → {'F': 16, 'K': (7, 7), 'S': 1, 'FUN': 'relu', 'POOL': True}

🧩 Entrenando modelo 6/10 → {'F': 64, 'K': (7, 7), 'S': 1, 'FUN': 'relu', 'POOL': True}

🧩 Entrenando modelo 7/10 → {'F': 64, 'K': (3, 3), 'S': 2, 'FUN': 'relu', 'POOL': False}

🧩 Entrenando modelo 8/10 → {'F': 64, 'K': (3, 3), 'S': 3, 'FUN': 'relu', 'POOL': False}

🧩 Entrenando modelo 9/10 → {'F': 64, 'K': (3, 3), 'S': 1, 'FUN': 'tanh', 'POOL': True}

🧩 Entrenando modelo 10/10 → {'F': 64, 'K': (3, 3), 'S': 1, 'FUN': 'sigmoid', 'POOL': True}


,Filtros,Kernel,Stride,Activación,MaxPooling,Parámetros,Épocas,Acc Train,Acc Test
0,4,3x3,1,relu,Sí,6810,35,0.9858,0.9781
1,16,3x3,1,relu,Sí,27210,5,0.9795,0.9461
2,64,3x3,1,relu,Sí,108810,16,0.9963,0.9838
3,4,7x7,1,relu,Sí,5050,5,0.9751,0.9421
4,16,7x7,1,relu,Sí,20170,12,0.9919,0.9876
5,64,7x7,1,relu,Sí,80650,5,0.9889,0.9771
6,64,3x3,2,relu,No,108810,5,0.9821,0.9524
7,64,3x3,3,relu,No,52490,5,0.9661,0.9275
8,64,3x3,1,tanh,Sí,108810,5,0.9807,0.9511
9,64,3x3,1,sigmoid,Sí,108810,5,0.9120,0.8804
